In [1]:
import pandas as pd
import numpy as np

from collections import Counter
from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report, confusion_matrix

# =========================
# CONFIG
# =========================
DATA_PATH = "../data/raw/development.csv"
N_SPLITS = 5
RANDOM_STATE = 42
USE_ONLY_FIRST_FOLD = True
APPLY_RULE_IF_FOUND = True
RULE_PRIORITY = "best_purity_then_freq"
MIN_RULE_SUPPORT = 30

# =========================
# LOAD + TIMESTAMP DROP
# =========================
df = pd.read_csv(DATA_PATH)

df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
df = df[df["timestamp"].notna()].reset_index(drop=True)

print("Samples after timestamp drop:", len(df))

# =========================
# BASIC FIXES
# =========================
df["article"] = df["article"].fillna("").astype(str)
df["title"]   = df["title"].fillna("").astype(str)
df["source"]  = df["source"].fillna("").astype(str)

# =========================
# TEXT
# =========================
def build_text(df):
	return (df["title"] + " " + df["article"]).str.lower()

df["text"] = build_text(df)

# =========================
# NUMERIC FEATURES (BASE)
# =========================
df["n_tokens"] = df["article"].str.split().str.len()
df["title_len"] = df["title"].str.len()
df["article_len"] = df["article"].str.len()
df["title_ratio"] = df["title_len"] / (df["article_len"] + 1)

df["year"]  = df["timestamp"].dt.year
df["month"] = df["timestamp"].dt.month
df["dow"]   = df["timestamp"].dt.dayofweek

# =========================
# X / y
# =========================
X = df[
	["source", "text",
	 "n_tokens", "title_len", "article_len",
	 "title_ratio", "year", "month", "dow"]
]
y = df["label"].astype(int)

# =========================
# RULE TOKENIZATION
# =========================
def tokenize_for_rules(text):
	return text.split()

# =========================
# RULE MINING (TRAIN ONLY)
# =========================
def mine_pure_rules(texts, labels):
	from collections import defaultdict

	counts = defaultdict(lambda: Counter())

	for txt, y in zip(texts, labels):
		for tok in set(tokenize_for_rules(txt)):
			counts[tok][y] += 1

	rule_token_to_class = {}
	rule_meta = {}

	for tok, c in counts.items():
		total = sum(c.values())
		if total < MIN_RULE_SUPPORT:
			continue

		best_class, best_freq = c.most_common(1)[0]
		purity = best_freq / total

		if purity >= 0.90:
			rule_token_to_class[tok] = best_class
			rule_meta[tok] = (purity, total)

	return rule_token_to_class, rule_meta

# =========================
# APPLY RULES
# =========================
def apply_rules(texts, rule_token_to_class, rule_meta):
	rule_pred = np.full(len(texts), -1, dtype=int)
	matched_token = [None] * len(texts)

	for i, txt in enumerate(texts):
		toks = set(tokenize_for_rules(txt))
		hits = [t for t in toks if t in rule_token_to_class]
		if not hits:
			continue

		if RULE_PRIORITY == "best_purity_then_freq":
			hits.sort(
				key=lambda t: (rule_meta[t][0], rule_meta[t][1]),
				reverse=True
			)
		else:
			hits.sort(key=lambda t: rule_meta[t][1], reverse=True)

		best = hits[0]
		rule_pred[i] = int(rule_token_to_class[best])
		matched_token[i] = best

	return rule_pred, matched_token

# =========================
# MODEL (IDENTICO AL TUO)
# =========================
def make_model():
	pre = ColumnTransformer(
		transformers=[
			("src", OneHotEncoder(handle_unknown="ignore"), ["source"]),

			("w_tfidf", TfidfVectorizer(
				analyzer="word",
				ngram_range=(1,2),
				min_df=3,
				max_df=0.9,
				sublinear_tf=True,
				max_features=250_000
			), "text"),

			("c_tfidf", TfidfVectorizer(
				analyzer="char_wb",
				ngram_range=(3,5),
				min_df=3,
				max_df=0.9,
				sublinear_tf=True,
				max_features=300_000
			), "text"),

			("num", StandardScaler(), [
				"n_tokens", "title_len", "article_len",
				"title_ratio", "year", "month", "dow"
			])
		],
		remainder="drop",
		n_jobs=-1
	)

	clf = LogisticRegression(
		C=2.0,
		class_weight="balanced",
		max_iter=2000,
		n_jobs=-1
	)

	return Pipeline([
		("pre", pre),
		("clf", clf)
	])

# =========================
# RUN (1 FOLD)
# =========================
skf = StratifiedKFold(
	n_splits=N_SPLITS,
	shuffle=True,
	random_state=RANDOM_STATE
)

for fold_id, (tr, te) in enumerate(skf.split(X, y), start=1):

	print(f"\n===== FOLD {fold_id} =====")

	X_tr, y_tr = X.iloc[tr], y.iloc[tr]
	X_te, y_te = X.iloc[te], y.iloc[te]

	# ---- Mine rules on TRAIN ONLY
	rule_token_to_class, rule_meta = mine_pure_rules(
		X_tr["text"], y_tr
	)
	print("Mined rules:", len(rule_token_to_class))

	# ---- Train model
	model = make_model()
	model.fit(X_tr, y_tr)

	# ---- Base predictions
	model_pred = model.predict(X_te)

	# ---- Apply rules
	rule_pred, matched_token = apply_rules(
		X_te["text"], rule_token_to_class, rule_meta
	)

	final_pred = model_pred.copy()
	mask = rule_pred != -1
	final_pred[mask] = rule_pred[mask]

	# ---- Metrics
	print("Macro F1:",
		f1_score(y_te, final_pred, average="macro"))

	print("\nConfusion Matrix:\n",
		confusion_matrix(y_te, final_pred))

	print("\nReport:\n",
		classification_report(y_te, final_pred, digits=3))

	print(f"Rule coverage: {mask.mean():.3f} ({mask.sum()} / {len(mask)})")
	if mask.any():
		print("Rule-only macro F1:",
			f1_score(y_te[mask], final_pred[mask], average="macro"))

	counter = Counter([t for t in matched_token if t is not None])
	print("Top matched rule tokens:",
		counter.most_common(20))

	if USE_ONLY_FIRST_FOLD:
		break


Samples after timestamp drop: 52247

===== FOLD 1 =====
Mined rules: 115
Macro F1: 0.747514418020134

Confusion Matrix:
 [[2194  103   72  153   32  469   59]
 [  59 1183   79   44   19   48   29]
 [  54   82 1475   53    9   37   30]
 [  82   54   59  780   56  125   30]
 [  10    9    0   29  891   28    2]
 [ 334   55   14  102   37 1039   46]
 [  28   11   14   12    3   24  293]]

Report:
               precision    recall  f1-score   support

           0      0.795     0.712     0.751      3082
           1      0.790     0.810     0.800      1461
           2      0.861     0.848     0.854      1740
           3      0.665     0.658     0.661      1186
           4      0.851     0.920     0.884       969
           5      0.587     0.639     0.612      1627
           6      0.599     0.761     0.670       385

    accuracy                          0.752     10450
   macro avg      0.735     0.764     0.748     10450
weighted avg      0.756     0.752     0.753     10450

Rule 